# **Day 81: NumPy Basics - A Satellite Image is an Array**

Phase 5 begins! We now learn the libraries that make Python powerful for **data** and **Remote Sensing**.

**NumPy** (Numerical Python) is the most important library for working with numbers. Almost every data and GIS library (Pandas, Matplotlib, Rasterio, scikit-learn) is built on top of it.

Install it with `pip install numpy`.

## **What is a NumPy Array?**
A NumPy **array** is a **grid of numbers**, all of the **same type**.
- A **1D array** is like a single row of numbers.
- A **2D array** is like a table (rows and columns).
- A **3D array** is like a stack of tables.

**The big connection to Remote Sensing:**
- One **band** of a satellite image is a **2D array** (rows x columns of pixels).
- A **multi-band image** is a **3D array** (bands x rows x columns).

In [1]:
import numpy as np
print(np.__version__)

2.5.3


In [2]:
temps = np.array([31, 33, 36, 34, 29, 35, 38])
print(temps)
print("Type:", type(temps))
print("Data type of values:", temps.dtype)
print("Shape:", temps.shape)
print("Dimensions:", temps.ndim)

[31 33 36 34 29 35 38]
Type: <class 'numpy.ndarray'>
Data type of values: int64
Shape: (7,)
Dimensions: 1


## **Why Not Just Use Lists?**
With a list, we need a loop to change every value. With NumPy, we write the maths **once**, and it is applied to **every value automatically**. This is called **vectorization**, and it is also much faster.

In [3]:
temps_list = [31, 33, 36, 34, 29, 35, 38]
print([t * 9 / 5 + 32 for t in temps_list])     # List: needs a loop
print(temps * 9 / 5 + 32)                        # NumPy: no loop needed

[87.8, 91.4, 96.8, 93.2, 84.2, 95.0, 100.4]
[ 87.8  91.4  96.8  93.2  84.2  95.  100.4]


In [4]:
big_list = list(range(1_000_000))
big_array = np.arange(1_000_000)
%timeit [x * 2 for x in big_list]
%timeit big_array * 2

37.2 ms ± 596 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


1.47 ms ± 6.95 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## **Creating Arrays**

In [5]:
print(np.zeros(5))                     # Five zeros
print(np.ones((2, 3), dtype=int))      # 2 rows x 3 columns of ones
print(np.full((2, 2), 7))              # Filled with 7
print(np.arange(0, 20, 5))             # Like range()
print(np.linspace(0, 1, 5))            # 5 equally spaced numbers from 0 to 1

[0. 0. 0. 0. 0.]
[[1 1 1]
 [1 1 1]]
[[7 7]
 [7 7]]
[ 0  5 10 15]
[0.   0.25 0.5  0.75 1.  ]


In [6]:
rng = np.random.default_rng(42)            # A random number generator with a fixed seed
print(rng.integers(1, 7, size=10))         # 10 dice rolls
print(rng.normal(30, 3, size=5).round(1))  # 5 temperatures around 30

[1 5 4 3 3 6 1 5 2 1]
[26.1 30.4 29.1 29.9 27.4]


## **A Satellite Band as a 2D Array**
Here is a tiny 5 x 6 piece of the **NIR band** of an image. Each number is the reflectance of one pixel.

In [7]:
nir = np.array([
    [0.02, 0.03, 0.35, 0.42, 0.45, 0.44],
    [0.02, 0.04, 0.38, 0.46, 0.47, 0.43],
    [0.03, 0.05, 0.30, 0.41, 0.44, 0.40],
    [0.21, 0.22, 0.24, 0.25, 0.39, 0.41],
    [0.20, 0.23, 0.22, 0.26, 0.24, 0.38],
])
print("Shape (rows, columns):", nir.shape)
print("Number of pixels:", nir.size)

Shape (rows, columns): (5, 6)
Number of pixels: 30


## **Indexing and Slicing a 2D Array**
We use `array[row, column]`. Slicing works on both rows and columns.

In [8]:
print(nir[0, 2])           # Row 0, column 2
print(nir[1])              # The whole of row 1
print(nir[:, 0])           # The whole of column 0
print(nir[0:2, 2:5])       # A small window: rows 0-1, columns 2-4

0.35
[0.02 0.04 0.38 0.46 0.47 0.43]
[0.02 0.02 0.03 0.21 0.2 ]
[[0.35 0.42 0.45]
 [0.38 0.46 0.47]]


Taking a small window of an image like `nir[0:2, 2:5]` is called **subsetting** or **clipping** in Remote Sensing.

## **The axis Idea**
- `axis=0` works **down the rows** (gives one answer per column).
- `axis=1` works **across the columns** (gives one answer per row).

In [9]:
print("Mean of whole band :", nir.mean().round(3))
print("Mean of each column:", nir.mean(axis=0).round(3))
print("Mean of each row   :", nir.mean(axis=1).round(3))
print("Max, Min           :", nir.max(), nir.min())

Mean of whole band : 0.28
Mean of each column: [0.096 0.114 0.298 0.36  0.398 0.412]
Mean of each row   : [0.285 0.3   0.272 0.287 0.255]
Max, Min           : 0.47 0.02


## **Boolean Masking: Finding Pixels That Match a Condition**
A **mask** is an array of True/False values. It is one of the most used ideas in Remote Sensing, for example to find water pixels or cloud pixels.

In [10]:
water_mask = nir < 0.1          # Water reflects very little NIR
print(water_mask)
print("Water pixels:", water_mask.sum())
print("Their values:", nir[water_mask])

[[ True  True False False False False]
 [ True  True False False False False]
 [ True  True False False False False]
 [False False False False False False]
 [False False False False False False]]
Water pixels: 6
Their values: [0.02 0.03 0.02 0.04 0.03 0.05]


In [11]:
# Combine conditions with & (and) and | (or). Brackets are needed.
vegetation = (nir > 0.35) & (nir < 0.50)
print("Vegetation pixels:", vegetation.sum())

# Change values using a mask: set water pixels to 0
cleaned = nir.copy()
cleaned[water_mask] = 0
print(cleaned)

Vegetation pixels: 13
[[0.   0.   0.35 0.42 0.45 0.44]
 [0.   0.   0.38 0.46 0.47 0.43]
 [0.   0.   0.3  0.41 0.44 0.4 ]
 [0.21 0.22 0.24 0.25 0.39 0.41]
 [0.2  0.23 0.22 0.26 0.24 0.38]]


Let us understand the code above:
- `nir < 0.1` compares **every** pixel with 0.1 and gives a True/False array of the same shape.
- `water_mask.sum()` counts the True values, because True counts as 1.
- `nir[water_mask]` picks only the pixels where the mask is True.

## **Data Types (dtype)**
Satellite images are usually stored as whole numbers to save space. For example, Sentinel-2 stores reflectance x 10,000 as **uint16** (whole numbers from 0 to 65,535).

In [12]:
raw = np.array([[523, 611], [3800, 4102]], dtype=np.uint16)
print(raw.dtype, raw.itemsize, "bytes per pixel")

reflectance = raw / 10000              # Division changes it to float
print(reflectance, reflectance.dtype)

uint16 2 bytes per pixel
[[0.0523 0.0611]
 [0.38   0.4102]] float64


## **Reshape and Stack: Building a Multi-Band Image**

In [13]:
a = np.arange(12)
print(a.reshape(3, 4))

# Three bands of the same 2 x 3 area, stacked into one 3D image
red = np.array([[0.05, 0.06, 0.20], [0.04, 0.05, 0.22]])
green = np.array([[0.08, 0.09, 0.18], [0.07, 0.08, 0.20]])
blue = np.array([[0.04, 0.05, 0.15], [0.04, 0.04, 0.16]])

image = np.stack([red, green, blue])
print("Image shape (bands, rows, columns):", image.shape)
print("Pixel at row 1, column 2 in all bands:", image[:, 1, 2])

[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
Image shape (bands, rows, columns): (3, 2, 3)
Pixel at row 1, column 2 in all bands: [0.22 0.2  0.16]


The last line gives the **spectral values** of one pixel across all bands. This is called the **spectral signature** of the pixel, and different materials (water, soil, plants) have different signatures.

In [14]:
# Quick question
# What is the shape of np.zeros((4, 100, 200))? How many bands, rows and columns?